# F.A.K.E.D. — Improved, classification-first five-architecture experiment (v3)

**Goal: maximize honestly measured real/fake performance while retaining manipulation-region pseudo-mask outputs**. Nothing in this notebook targets or guarantees 85% accuracy; training and evaluation must determine achieved performance.

### What is improved over the previous five-model notebook

1. **More varied training examples**: use up to 240 real/fake source pairs and 8 matched frames/pair rather than 100 × 6, while retaining source-identity-disjoint splits. Dataset counts are recorded exactly.
2. **Two-phase optimization**: first train the **classification objective alone**, then introduce localization gradually. The pseudo-mask decoder sees detached encoder features: *no noisy pseudo-mask gradients update the classification encoder*. After restoring the best classifier checkpoint, a short decoder-only refinement stage trains segmentation without modifying the classifier.
3. **Artifact-preserving augmentation**: mild flips/photometric changes only; blur, JPEG re-encoding and random zoom disabled by default.
4. **More patient training**: longer schedule, conservative pretrained-backbone LR, checkpoint selected on validation ROC-AUC with video-level accuracy as a tie-breaker. Per-epoch train/validation curves.
5. **Real-face-aware localization**: optimize segmentation thresholds against fake pseudo-IoU **and** real-face false-mask area; save both raw heatmaps and **class-gated segmentation overlays**.
6. **Leakage auditing and report export**: split tables, method distributions, paired real/fake input examples, same test faces with outputs from all models, confusion matrices, ROC charts, precision/recall/F1, comparison CSVs, per-class examples, checkpoints and ZIP.
7. **Validation only for tuning**: test set used only for final estimates. Never invent accuracy or describe pseudo-reference labels as manual masks.

### Kaggle: GPU accelerator ON; dataset attached; internet ON once for ImageNet pretrained model weights (or cached weights).

**Run once with `SMOKE_TEST=True`**, then set it to False and restart **Run All** for actual training. For a quick diagnosis, run only `M5_MobileNetV3Large_UNet` first by changing `MODEL_TO_RUN`; for report comparisons, run all five on the same split. The notebook creates a distinct experiment folder whenever data/training settings change. Failed runs can resume completed models within that folder.


In [1]:
import os, sys, time, random, re, gc, json, math, warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings("ignore")

NOTEBOOK_START = time.time()
elapsed = lambda: time.time() - NOTEBOOK_START
def hms(s):
    s = max(0, int(s))
    return f"{s//3600:02d}:{(s%3600)//60:02d}:{s%60:02d}"

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, roc_auc_score,
                             roc_curve, confusion_matrix, classification_report,
                             balanced_accuracy_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

IN_KAGGLE = Path("/kaggle").exists()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

def amp_ctx():
    return torch.autocast(device_type=DEVICE.type, enabled=USE_AMP)

print(f"torch {torch.__version__} | device: {DEVICE} | AMP: {USE_AMP}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
from IPython.display import display


torch 2.11.0+cu128 | device: cuda | AMP: True
GPU: Tesla T4


## 1. Settings and experiment scale

Default: **100 source pairs × up to 6 matched frames × 2 labels ≈ 1,200 face crops** (minus unreadable frames / detector misses). **10 epochs max/model** with early stopping. This is a computationally bounded experiment, not full FF++ training. Runtime will vary; the five architectures execute sequentially. Lower counts for a faster test. All outputs include actual sample counts.


In [2]:
# ------------------------------- PATHS ---------------------------------
CANDIDATE_DATA_ROOTS = [
    Path("/kaggle/input/datasets/xdxd003"),
    Path("/kaggle/input")
]

def find_data_root():
    for root in CANDIDATE_DATA_ROOTS:
        if not root.exists():
            continue
        if (root / "original").exists():
            return root
        for p in root.glob("**/original"):
            return p.parent
    return CANDIDATE_DATA_ROOTS[0]

DATA_ROOT = find_data_root()
BASE_ROOT = (Path("/kaggle/working") if IN_KAGGLE else Path("./working")) / "FAKED_STUDENT"
WORK_ROOT = BASE_ROOT / "initializing"

FACE_ROOT = WORK_ROOT / "faces"
MASK_ROOT = WORK_ROOT / "masks"
CKPT_DIR = WORK_ROOT / "checkpoints"
PLOT_DIR = WORK_ROOT / "plots"

for d in (FACE_ROOT / "real", FACE_ROOT / "fake", MASK_ROOT, CKPT_DIR, PLOT_DIR):
    d.mkdir(parents=True, exist_ok=True)

REAL_METHOD = "original"
FAKE_METHODS_KNOWN = [
    "Deepfakes",
    "Face2Face",
    "FaceSwap",
    "NeuralTextures",
    "FaceShifter",
    "DeepFakeDetection"
]

# ---------------------------- DATA SCALE -------------------------------
N_SOURCE_VIDEOS  = 1000
FRAMES_PER_VIDEO = 12
MAX_FRAME_SCAN   = 350
IMAGE_SIZE       = 224

# ---------------------------- OPTIMISATION -----------------------------
BATCH_SIZE       = 16
EPOCHS           = 22
WARMUP_EPOCHS    = 1

LR_ENCODER       = 3e-5
LR_HEAD          = 3e-4
WEIGHT_DECAY     = 5e-5
LABEL_SMOOTHING  = 0.01
GRAD_CLIP        = 1.0
LEAKY_RELU_SLOPE = 0.01

SEG_WEIGHT       = 0.12
DICE_WEIGHT      = 0.50

CLS_ONLY_EPOCHS       = 6
EARLY_STOP_PATIENCE   = 7
SEG_REFINE_EPOCHS     = 5
SEG_REFINE_LR         = 4e-4
MIN_TRAIN_EPOCHS      = 11
REAL_MASK_BCE_WEIGHT  = 2.0
SAVE_QUALITY_CONTROL  = True

# ---------------------------- PREPROCESSING ----------------------------
FACE_PADDING     = 0.25
DETECTION_WIDTH  = 480
JPEG_QUALITY     = 95

MASK_MIN_DIFF    = 14
MASK_MIN_AREA    = 0.004
MASK_MAX_AREA    = 0.65

TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15

NUM_WORKERS = 2 if IN_KAGGLE else 0

# ---------------------------- TIME BUDGET ------------------------------
TOTAL_TIME_BUDGET_SEC      = 24 * 60 * 60
PREPROCESS_TIME_BUDGET_SEC = 180 * 60
TRAIN_TIME_BUDGET_SEC      = 300 * 60

time_left = lambda: TOTAL_TIME_BUDGET_SEC - elapsed()

# --------------------------- PROJECT SWITCHES --------------------------
SMOKE_TEST = False

MODEL_TO_RUN = [
    "M5_MobileNetV3Large_UNet",
    "M1_ResNet18_UNet",
    "M2_EfficientNetB0_UNet",
    "M3_ResNet18_AttentionUNet",
    "M4_EfficientNetB0_AttentionUNet"
]

USE_PRETRAINED = True
ENABLE_TTA = True

# ----------------------------- SMOKE TEST ------------------------------
if SMOKE_TEST:
    N_SOURCE_VIDEOS = 15
    FRAMES_PER_VIDEO = 2
    EPOCHS = 1
    TRAIN_TIME_BUDGET_SEC = 15 * 60
    CLS_ONLY_EPOCHS = 0
    MIN_TRAIN_EPOCHS = 1

assert N_SOURCE_VIDEOS >= 12

BASE_ROOT.mkdir(parents=True, exist_ok=True)

# -------------------------- EXPERIMENT INFO ----------------------------
TOTAL_CROPS = N_SOURCE_VIDEOS * FRAMES_PER_VIDEO * 2

print(f"DATA_ROOT : {DATA_ROOT}")
print(f"WORK_ROOT : {WORK_ROOT}")

print(f"Target    : {N_SOURCE_VIDEOS} pairs x {FRAMES_PER_VIDEO} frames")
print(f"Total     : ~{TOTAL_CROPS:,} face crops at {IMAGE_SIZE}px")

print(f"Training  : {EPOCHS} epochs")
print(f"Batch     : {BATCH_SIZE}")
print(f"LR        : Encoder={LR_ENCODER}, Head={LR_HEAD}")

print(f"Models    : {len(MODEL_TO_RUN)}")
print(f"Preprocess budget : {PREPROCESS_TIME_BUDGET_SEC // 3600} hours")
print(f"Training budget   : {TRAIN_TIME_BUDGET_SEC // 3600} hours")

print(
    f"Running {'SMOKE TEST' if SMOKE_TEST else 'REAL EXPERIMENT'}; "
    f"requested {N_SOURCE_VIDEOS} pairs x {FRAMES_PER_VIDEO} frames"
)

DATA_ROOT : /kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23
WORK_ROOT : /kaggle/working/FAKED_STUDENT/initializing
Target    : 1000 pairs x 12 frames
Total     : ~24,000 face crops at 224px
Training  : 22 epochs
Batch     : 16
LR        : Encoder=3e-05, Head=0.0003
Models    : 5
Preprocess budget : 3 hours
Training budget   : 5 hours
Running REAL EXPERIMENT; requested 1000 pairs x 12 frames


## 2. Dataset discovery, pairing, and reproducible experiment ID

Only videos in the attached dataset are indexed. Every real source is paired with one available manipulated clip. We keep all frames from the same source and its manipulated variants in the same split. The saved experiment ID changes when sample selection, preprocessing or hyperparameters change, preventing stale metrics from being reused accidentally.


In [3]:
assert DATA_ROOT.exists(), f"Dataset root not found: {DATA_ROOT}"
video_paths = sorted(DATA_ROOT.rglob("*.mp4"))
assert video_paths, f"No MP4 videos found under {DATA_ROOT}"

records = []
for vp in video_paths:
    method = vp.parent.name
    if method != REAL_METHOD and method not in FAKE_METHODS_KNOWN:
        continue
    ids = re.findall(r"\d+", vp.stem)
    records.append({
        "video_id": vp.stem,
        "video_path": str(vp),
        "label": 0 if method == REAL_METHOD else 1,
        "class_name": "real" if method == REAL_METHOD else "fake",
        "manipulation_method": method,
        "primary_source_id": ids[0] if ids else vp.stem,
    })

video_manifest = pd.DataFrame(records).drop_duplicates("video_path").reset_index(drop=True)
real_manifest = video_manifest[video_manifest.class_name == "real"].reset_index(drop=True)
fake_manifest = video_manifest[video_manifest.class_name == "fake"].reset_index(drop=True)
methods_available = sorted(fake_manifest.manipulation_method.unique())

print(f"Videos indexed: {len(video_manifest):,}  "
      f"(real {len(real_manifest):,} | fake {len(fake_manifest):,})")
print(f"Fake methods  : {methods_available}")

# ---- pair each real video with one fake derived from it ----
fake_lookup = defaultdict(dict)
for _, row in fake_manifest.iterrows():
    fake_lookup[row["primary_source_id"]][row["manipulation_method"]] = row

pairs, mi = [], 0
for _, r in real_manifest.sample(frac=1.0, random_state=SEED).iterrows():
    candidates = fake_lookup.get(r["primary_source_id"])
    if not candidates:
        continue
    chosen = None
    for _ in range(len(methods_available)):                 # round-robin over methods
        m = methods_available[mi % len(methods_available)]
        mi += 1
        if m in candidates:
            chosen = candidates[m]
            break
    chosen = chosen if chosen is not None else next(iter(candidates.values()))
    pairs.append({"video_id": r["video_id"], "real_row": r, "fake_row": chosen})
    if len(pairs) >= N_SOURCE_VIDEOS:
        break

assert pairs, "No real/fake pairs formed — check source-id parsing for this dataset mirror."
print(f"\nPaired {len(pairs)} videos (requested {N_SOURCE_VIDEOS}).")
print(pd.Series([p["fake_row"]["manipulation_method"] for p in pairs])
      .value_counts().to_string())
# Hash actual selected file pairs and experiment configuration for safe reuse.
import hashlib
SETTINGS = {"selected_paths": [(str(x['real_row']['video_path']),str(x['fake_row']['video_path'])) for x in pairs],
            "seed":SEED,"source_pairs":N_SOURCE_VIDEOS,"frames_per_video":FRAMES_PER_VIDEO,
            "max_frame_scan":MAX_FRAME_SCAN,"resolution":IMAGE_SIZE,"face_padding":FACE_PADDING,
            "mask_min_diff":MASK_MIN_DIFF,"mask_min_area":MASK_MIN_AREA,"mask_max_area":MASK_MAX_AREA,
            "split":(TRAIN_FRAC,VAL_FRAC),"epochs":EPOCHS,"batch":BATCH_SIZE,
            "optimizer":(LR_ENCODER,LR_HEAD,WEIGHT_DECAY),"seg_weight":SEG_WEIGHT,
            "cls_only_epochs":CLS_ONLY_EPOCHS,"early_stop_patience":EARLY_STOP_PATIENCE,
            "seg_refine_epochs":SEG_REFINE_EPOCHS,"seg_refine_lr":SEG_REFINE_LR,
            "real_mask_bce_weight":REAL_MASK_BCE_WEIGHT,
            "models":MODEL_TO_RUN,"architecture_revision":"classification_first_decoder_refinement_v4",
            "smoke_test":SMOKE_TEST,"pretrained":USE_PRETRAINED}
RUN_ID = hashlib.sha256(json.dumps(SETTINGS,sort_keys=True).encode()).hexdigest()[:12]
WORK_ROOT = BASE_ROOT / "experiments" / RUN_ID
FACE_ROOT, MASK_ROOT = WORK_ROOT / "faces",WORK_ROOT / "masks"
REPORT_DIR, CKPT_DIR = WORK_ROOT / "report_artifacts",WORK_ROOT / "checkpoints"
for d in [FACE_ROOT/"real",FACE_ROOT/"fake",MASK_ROOT,REPORT_DIR,CKPT_DIR]:
    d.mkdir(parents=True,exist_ok=True)
(REPORT_DIR / "run_settings.json").write_text(json.dumps({"run_id":RUN_ID,**SETTINGS},indent=2))
video_manifest.to_csv(REPORT_DIR / "all_indexed_videos.csv",index=False)
PLOT_DIR=REPORT_DIR
pd.DataFrame([{"real_source_id":str(p['video_id']),
               "real_video_path":str(p['real_row']['video_path']),
               "fake_video_path":str(p['fake_row']['video_path']),
               "manipulation_method":str(p['fake_row']['manipulation_method'])} for p in pairs]).to_csv(
                   REPORT_DIR / "selected_video_pairs.csv",index=False)
print("Run ID:",RUN_ID,"Report output:",REPORT_DIR)


Videos indexed: 7,000  (real 1,000 | fake 6,000)
Fake methods  : ['DeepFakeDetection', 'Deepfakes', 'Face2Face', 'FaceShifter', 'FaceSwap', 'NeuralTextures']

Paired 1000 videos (requested 1000).
Deepfakes         200
Face2Face         200
FaceShifter       200
FaceSwap          200
NeuralTextures    200
Run ID: 14adbe7bd653 Report output: /kaggle/working/FAKED_STUDENT/experiments/14adbe7bd653/report_artifacts


## 3. Preprocess actual real and fake faces, plus pseudo-localization masks

Real and manipulated frames are read at matching indices, and **the face box is detected on the real frame and reused for the fake**. Two aligned crops are saved per successful frame. A morphological difference mask is constructed from the aligned real/fake pair and rejected if its coverage is implausible. These *pseudo-reference labels* are useful for a demonstration but not equivalent to manual ground truth. Crops are stored in the run directory for your real/fake visual examples.


In [4]:
face_cascade = cv2.CascadeClassifier(
    str(Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"))
assert not face_cascade.empty(), "OpenCV Haar cascade failed to load."


def frame_count_of(path):
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if cap.isOpened() else 0
    cap.release()
    return n


def read_frames_at(path, indices):
    """Open the clip ONCE and pull all requested frames in a single sequential pass.
    grab() advances the decoder without the colour conversion, so skipped frames are cheap.
    (The old version re-opened the file and re-sought for every single frame.)"""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        cap.release()
        return {}
    wanted = sorted({int(i) for i in indices})
    out, pos, w = {}, 0, 0
    while w < len(wanted):
        if not cap.grab():
            break
        if pos == wanted[w]:
            ok, frame = cap.retrieve()
            if ok:
                out[wanted[w]] = frame
            w += 1
        pos += 1
    cap.release()
    return out


def detect_face(frame):
    h, w = frame.shape[:2]
    scale = min(1.0, DETECTION_WIDTH / w)
    small = cv2.resize(frame, (max(1, int(w * scale)), max(1, int(h * scale))),
                       interpolation=cv2.INTER_AREA) if scale < 1.0 else frame
    gray = cv2.equalizeHist(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY))
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
    if len(faces) == 0:
        return None
    x, y, fw, fh = max(faces, key=lambda b: b[2] * b[3])
    inv = 1.0 / scale
    return int(x * inv), int(y * inv), int(fw * inv), int(fh * inv)


def padded_bounds(shape, box, padding=FACE_PADDING):
    x, y, w, h = box
    fh, fw = shape[:2]
    px, py = int(w * padding), int(h * padding)
    return (max(0, x - px), max(0, y - py), min(fw, x + w + px), min(fh, y + h + py))


def make_pseudo_mask(real_crop, fake_crop):
    """Pixel-level manipulation label from the aligned real/fake difference.
    Returns None when the difference is not a trustworthy localisation."""
    gray = cv2.cvtColor(cv2.absdiff(real_crop, fake_crop), cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)

    otsu_t, _ = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = ((gray >= max(MASK_MIN_DIFF, otsu_t)).astype(np.uint8)) * 255

    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8), iterations=2)

    n, lbl, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return None
    areas = stats[1:, cv2.CC_STAT_AREA]
    biggest = areas.max()
    keep = [i for i in range(1, n) if stats[i, cv2.CC_STAT_AREA] >= 0.30 * biggest]
    mask = (np.isin(lbl, keep).astype(np.uint8)) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))

    frac = float((mask > 0).mean())
    if not (MASK_MIN_AREA <= frac <= MASK_MAX_AREA):
        return None
    return mask


CACHE_MANIFEST = WORK_ROOT / "frame_manifest.csv"
if (CACHE_MANIFEST.exists() and (WORK_ROOT / "preprocess_complete.json").exists()):
    frame_manifest = pd.read_csv(CACHE_MANIFEST,dtype={"video_id":str,"sample_id":str,"clip_id":str})
    file_check = frame_manifest.image_path.dropna().astype(str).map(lambda p: Path(p).exists())
    if len(frame_manifest) and file_check.all():
        print("Loaded",len(frame_manifest),"previously preprocessed crops from this exact experiment.")
    else:
        raise RuntimeError("Preprocessing cache is incomplete. Delete the run folder and rerun.")
else:
    t0 = time.time()
    frame_records, stats_counter = [], defaultdict(int)

    for pair in tqdm(pairs, desc="Preprocessing pairs"):
        if time.time() - t0 > PREPROCESS_TIME_BUDGET_SEC:
            print(f"\nPreprocessing budget reached — stopping with "
                  f"{stats_counter['crops']} crops.")
            break

        real_path = Path(pair["real_row"]["video_path"])
        fake_path = Path(pair["fake_row"]["video_path"])
        method = pair["fake_row"]["manipulation_method"]

        n_frames = min(frame_count_of(real_path), frame_count_of(fake_path), MAX_FRAME_SCAN)
        if n_frames <= 0:
            stats_counter["skipped_pairs"] += 1
            continue

        idxs = np.unique(np.linspace(0, n_frames - 1, FRAMES_PER_VIDEO, dtype=int)).tolist()
        real_frames = read_frames_at(real_path, idxs)
        fake_frames = read_frames_at(fake_path, idxs)

        last_box = None
        for i in idxs:
            rf, ff = real_frames.get(i), fake_frames.get(i)
            if rf is None or ff is None or rf.shape != ff.shape:
                continue

            box = detect_face(rf) or last_box
            if box is None:
                stats_counter["no_face"] += 1
                continue
            last_box = box                                  # reuse on detector misses

            x1, y1, x2, y2 = padded_bounds(rf.shape, box)
            if x2 - x1 < 32 or y2 - y1 < 32:
                continue

            real_crop = cv2.resize(rf[y1:y2, x1:x2], (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
            fake_crop = cv2.resize(ff[y1:y2, x1:x2], (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)

            mask = make_pseudo_mask(real_crop, fake_crop)
            sid = f"{pair['video_id']}_{method}_{i:06d}"

            real_out = FACE_ROOT / "real" / f"{sid}.jpg"
            fake_out = FACE_ROOT / "fake" / f"{sid}.jpg"
            cv2.imwrite(str(real_out), real_crop, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            cv2.imwrite(str(fake_out), fake_crop, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

            if mask is not None:
                mask_out = MASK_ROOT / f"{sid}.png"
                cv2.imwrite(str(mask_out), mask)
                stats_counter["valid_masks"] += 1
            else:
                mask_out = None
                stats_counter["rejected_masks"] += 1

            common = {"sample_id": sid, "video_id": pair["video_id"], "manipulation_method": method}
            frame_records.append({**common, "clip_id": f"{pair['video_id']}__real",
                                  "image_path": str(real_out), "mask_path": None,
                                  "label": 0, "seg_valid": 0})
            frame_records.append({**common, "clip_id": f"{pair['video_id']}__fake",
                                  "image_path": str(fake_out),
                                  "mask_path": (str(mask_out) if mask_out else None),
                                  "label": 1, "seg_valid": int(mask is not None)})
            stats_counter["crops"] += 2

    assert stats_counter["crops"] > 0, "No crops produced — inspect dataset layout / detector."

    frame_manifest = pd.DataFrame(frame_records)
    frame_manifest.to_csv(CACHE_MANIFEST, index=False)
    (WORK_ROOT / "preprocess_complete.json").write_text(json.dumps({"run_id":RUN_ID,"rows":len(frame_manifest)}))

    n_fake = int((frame_manifest.label == 1).sum())
    print(f"\nPreprocessing: {hms(time.time() - t0)}")
    print(f"Crops              : {len(frame_manifest):,} "
          f"(real {int((frame_manifest.label==0).sum()):,} | fake {n_fake:,})")
    print(f"Usable seg. masks  : {stats_counter['valid_masks']:,} / {n_fake:,} fakes "
          f"({100*stats_counter['valid_masks']/max(1,n_fake):.1f}%)")
    print(f"Rejected masks     : {stats_counter['rejected_masks']:,}   "
          f"detector misses: {stats_counter['no_face']}   skipped pairs: {stats_counter['skipped_pairs']}")
    print(f"Elapsed            : {hms(elapsed())} / {hms(TOTAL_TIME_BUDGET_SEC)}")
# Explicit counts for manuscript dataset subsection.
raw_count = video_manifest.groupby(["class_name","manipulation_method"]).size().reset_index(name="n_videos")
raw_count.to_csv(REPORT_DIR / "available_original_dataset_index.csv",index=False)
print("Actual processed frame crops:",len(frame_manifest),"valid fake pseudo-masks:",int(frame_manifest.seg_valid.sum()))


Preprocessing pairs:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 4. Face examples and mask-provenance figure

The first figure explicitly contains real faces and fake faces from the matched source, their difference-derived pseudo-reference masks, and paired examples. For input vs. predicted masks across all models, use the figure generated in §9 below.


In [ ]:
fakes_with_mask = frame_manifest[(frame_manifest.label == 1) & (frame_manifest.seg_valid == 1)]
assert len(fakes_with_mask), ("No usable pseudo-masks were produced. Lower MASK_MIN_DIFF or widen "
                              "MASK_MIN_AREA/MASK_MAX_AREA, then re-run preprocessing.")
show = fakes_with_mask.sample(n=min(4, len(fakes_with_mask)), random_state=SEED)

fig, axes = plt.subplots(len(show), 4, figsize=(13, 3.1 * len(show)))
axes = np.atleast_2d(axes)
for r, (_, row) in enumerate(show.iterrows()):
    fake = cv2.cvtColor(cv2.imread(row["image_path"]), cv2.COLOR_BGR2RGB)
    real = cv2.cvtColor(cv2.imread(row["image_path"].replace("/fake/", "/real/")), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
    diff = cv2.cvtColor(cv2.applyColorMap(
        cv2.convertScaleAbs(cv2.absdiff(real, fake), alpha=4.0), cv2.COLORMAP_INFERNO),
        cv2.COLOR_BGR2RGB)
    for c, (im, ttl, kw) in enumerate([
            (real, "Real crop", {}), (fake, f"Fake ({row['manipulation_method']})", {}),
            (diff, "|real - fake| (x4)", {}), (mask, "Pseudo GT mask", {"cmap": "gray"})]):
        axes[r, c].imshow(im, **kw); axes[r, c].set_title(ttl, fontsize=10); axes[r, c].axis("off")
fig.suptitle("Where the segmentation labels come from", y=1.005, fontsize=13)
fig.tight_layout(); fig.savefig(PLOT_DIR / "mask_provenance.png", dpi=110, bbox_inches="tight")
plt.show()

areas = [float((cv2.imread(p, cv2.IMREAD_GRAYSCALE) > 127).mean())
         for p in fakes_with_mask["mask_path"].sample(
             n=min(400, len(fakes_with_mask)), random_state=SEED)]
fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(areas, bins=40, color="#4C72B0", edgecolor="white")
ax.set_xlabel("fraction of crop marked manipulated"); ax.set_ylabel("masks")
ax.set_title(f"Mask area distribution (median {np.median(areas):.1%})")
fig.tight_layout(); fig.savefig(PLOT_DIR / "mask_area_hist.png", dpi=110); plt.show()
# Report-ready representative real, fake and pseudo-mask images (same aligned pairs).
paired = frame_manifest.sort_values(["sample_id","label"])
sample_ids = paired[(paired.label==1) & (paired.seg_valid==1)].sample(
    n=min(4,int(((paired.label==1)&(paired.seg_valid==1)).sum())),random_state=SEED).sample_id.tolist()
fig,axes=plt.subplots(max(1,len(sample_ids)),4,figsize=(12,3.1*max(1,len(sample_ids))),squeeze=False)
for i,sid in enumerate(sample_ids):
    subset=paired[paired.sample_id==sid].set_index('label')
    real=cv2.cvtColor(cv2.imread(str(subset.loc[0,'image_path'])),cv2.COLOR_BGR2RGB)
    fake=cv2.cvtColor(cv2.imread(str(subset.loc[1,'image_path'])),cv2.COLOR_BGR2RGB)
    mask=cv2.imread(str(subset.loc[1,'mask_path']),cv2.IMREAD_GRAYSCALE)
    diff=cv2.absdiff(real,fake)
    for j,(im,title,cmap) in enumerate([(real,'Real face crop',None),(fake,'Manipulated face crop',None),
                                         (diff,'Absolute RGB difference',None),
                                         (mask,'Difference-derived pseudo-mask','gray')]):
        axes[i,j].imshow(im,cmap=cmap);axes[i,j].set_title(title,fontsize=12,fontweight='bold')
        axes[i,j].axis('off')
fig.tight_layout();fig.savefig(REPORT_DIR/'REAL_FAKE_FACE_CROPS_AND_PSEUDO_MASKS.png',dpi=190,bbox_inches='tight')
plt.show();plt.close(fig)


## 5. Leakage-aware split, augmentation, and shared dataloaders

We split by **connected source identity components**, not by individual frames. If a selected manipulation uses source IDs `001_002`, all examples sharing `001` or `002` are grouped together before train/validation/test assignment. Augmentation is applied **only to the training samples**. The resulting CSV records the exact data counts used.


In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class FacedDataset(Dataset):
    def __init__(self, df, train=False, size=IMAGE_SIZE):
        self.df = df.reset_index(drop=True)
        self.train, self.size = train, size

    def __len__(self):
        return len(self.df)

    def _augment(self, img, mask):
        # Avoid blur, JPEG re-encoding or tight random crops: these can destroy
        # precisely the high-frequency traces needed to detect manipulated video.
        if random.random() < 0.5:
            img, mask = cv2.flip(img, 1), cv2.flip(mask, 1)
        if random.random() < 0.3:
            alpha = random.uniform(0.96, 1.04)
            beta = random.uniform(-5., 5.)
            img = np.clip(img.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        return img, mask

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = cv2.imread(row["image_path"])
        if img is None:
            raise FileNotFoundError(f"Missing face crop: {row['image_path']}")

        mp = row["mask_path"]
        if isinstance(mp, str) and mp:
            m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
            mask = (m > 127).astype(np.float32) if m is not None else np.zeros((self.size, self.size), np.float32)
        else:
            mask = np.zeros((self.size, self.size), np.float32)

        if self.train:
            img, mask = self._augment(img, mask)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        return (torch.from_numpy(img).permute(2, 0, 1).float(),
                torch.tensor(int(row["label"]), dtype=torch.long),
                torch.from_numpy(mask).unsqueeze(0).float(),
                torch.tensor(float(row["seg_valid"]), dtype=torch.float32))


# ---- group-aware split ----
# Connected-components grouping: any fake containing multiple source IDs ties those IDs together.
source_map={}
for pair in pairs:
    vid=str(pair["video_id"])
    srcs=set(re.findall(r"\d+",str(pair["fake_row"]["video_id"])))
    srcs.add(vid)
    source_map[vid]=srcs
parent={v:v for v in source_map}
def _root(v):
    while parent[v]!=v:
        parent[v]=parent[parent[v]];v=parent[v]
    return v
def _union(a,b):
    a,b=_root(a),_root(b)
    if a!=b:parent[b]=a
owner={}
for vid,src_ids in source_map.items():
    for sid in src_ids:
        if sid in owner:_union(vid,owner[sid])
        else:owner[sid]=vid
component_of={v:_root(v) for v in source_map}
vids=sorted({component_of[str(v)] for v in frame_manifest.video_id.unique()})
random.Random(SEED).shuffle(vids)
n_tr = max(1, int(len(vids) * TRAIN_FRAC))
n_va = max(1, int(len(vids) * VAL_FRAC))
split_of = {}
for v in vids[:n_tr]:            split_of[v] = "train"
for v in vids[n_tr:n_tr + n_va]: split_of[v] = "val"
for v in vids[n_tr + n_va:]:     split_of[v] = "test"
frame_manifest["split"] = frame_manifest["video_id"].astype(str).map(lambda v: split_of[component_of[v]])
assert frame_manifest.split.notna().all()
assert len(vids)>=3,"Need at least three source-disjoint groups. Increase N_SOURCE_VIDEOS."
frame_manifest.to_csv(REPORT_DIR / "shared_frame_manifest.csv",index=False)

train_df = frame_manifest[frame_manifest.split == "train"].reset_index(drop=True)
val_df   = frame_manifest[frame_manifest.split == "val"].reset_index(drop=True)
test_df  = frame_manifest[frame_manifest.split == "test"].reset_index(drop=True)
assert len(train_df) and len(val_df) and len(test_df), "Empty split — raise N_SOURCE_VIDEOS."

loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
if NUM_WORKERS > 0:
    loader_kw.update(persistent_workers=True, prefetch_factor=4)

train_dl = DataLoader(FacedDataset(train_df, train=True),  batch_size=BATCH_SIZE,
                      shuffle=True, drop_last=False, **loader_kw)
val_dl   = DataLoader(FacedDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, **loader_kw)
test_dl  = DataLoader(FacedDataset(test_df),  batch_size=BATCH_SIZE, shuffle=False, **loader_kw)

print(pd.DataFrame([{
    "split": s, "videos": d.video_id.nunique(), "crops": len(d),
    "real": int((d.label == 0).sum()), "fake": int((d.label == 1).sum()),
    "with mask": int(d.seg_valid.sum()),
} for s, d in [("train", train_df), ("val", val_df), ("test", test_df)]]).to_string(index=False))
print(f"\nSteps per epoch: {len(train_dl)}")
# Split audit: fail if the same source identity appears in multiple splits.
for split_a,split_b in [('train','val'),('train','test'),('val','test')]:
    a=set(frame_manifest[frame_manifest.split==split_a].video_id.astype(str))
    b=set(frame_manifest[frame_manifest.split==split_b].video_id.astype(str))
    assert not a.intersection(b)
    src_a=set().union(*(source_map[v] for v in a)) if a else set()
    src_b=set().union(*(source_map[v] for v in b)) if b else set()
    assert not src_a.intersection(src_b),f"Source leakage between {split_a} and {split_b}"
assert train_df.label.nunique()==2 and val_df.label.nunique()==2 and test_df.label.nunique()==2
split_counts=pd.DataFrame([{'split':name,'source_pairs':int(d.video_id.nunique()),
                            'real_frames':int((d.label==0).sum()),'fake_frames':int((d.label==1).sum()),
                            'total_frames':len(d),'valid_fake_pseudo_masks':int(d.seg_valid.sum())}
     for name,d in [('train',train_df),('validation',val_df),('test',test_df)]])
split_counts.to_csv(REPORT_DIR/'train_validation_test_counts.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].bar(split_counts.split,split_counts.real_frames,label='Real frames')
axes[0].bar(split_counts.split,split_counts.fake_frames,bottom=split_counts.real_frames,label='Fake frames')
axes[0].set(title='Processed train / validation / test crops',ylabel='Frames');axes[0].legend()
methods=frame_manifest[frame_manifest.label==1].manipulation_method.value_counts()
axes[1].barh(methods.index[::-1],methods.values[::-1])
axes[1].set(title='Fake frame count by manipulation method',xlabel='Frames')
fig.tight_layout();fig.savefig(REPORT_DIR/'DATASET_DISTRIBUTION_AND_SPLIT.png',dpi=180,bbox_inches='tight');plt.show();plt.close(fig)


## 6. Five model architectures (real backbone differences + attention gating)

Each image enters a pretrained feature encoder. A classification head produces a real/fake probability, and a skip-connected decoder produces a full-resolution manipulation probability map. The models differ in encoder family and whether attention gates filter decoder skip features.


In [ ]:
from torchvision.models import (resnet18, ResNet18_Weights, efficientnet_b0,
                                EfficientNet_B0_Weights, mobilenet_v3_large,
                                MobileNet_V3_Large_Weights)
from torchvision.models.feature_extraction import create_feature_extractor

# Feature taps: validated against standard torchvision model layer naming.
MODEL_SPECS = {
    "M1_ResNet18_UNet": ("resnet18", False),
    "M2_EfficientNetB0_UNet": ("efficientnet_b0", False),
    "M3_ResNet18_AttentionUNet": ("resnet18", True),
    "M4_EfficientNetB0_AttentionUNet": ("efficientnet_b0", True),
    "M5_MobileNetV3Large_UNet": ("mobilenet_v3_large", False),
}

def make_backbone(kind, pretrained=True):
    if kind == "resnet18":
        net=resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        nodes={"relu":"s1","layer1":"s2","layer2":"s3","layer3":"s4","layer4":"s5"}
    elif kind == "efficientnet_b0":
        net=efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT if pretrained else None)
        nodes={"features.1":"s1","features.2":"s2","features.3":"s3",
               "features.5":"s4","features.8":"s5"}
    elif kind == "mobilenet_v3_large":
        net=mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT if pretrained else None)
        nodes={"features.1":"s1","features.3":"s2","features.6":"s3",
               "features.12":"s4","features.16":"s5"}
    else:raise ValueError(kind)
    extractor=create_feature_extractor(net,return_nodes=nodes)
    with torch.no_grad():
        z=extractor(torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE))
    channels=[int(z[f"s{i}"].shape[1]) for i in range(1,6)]
    sizes=[int(z[f"s{i}"].shape[-1]) for i in range(1,6)]
    assert sizes==sorted(sizes,reverse=True), (kind,sizes)
    return extractor,channels

class DoubleConv(nn.Module):
    def __init__(self,ci,co):
        super().__init__()
        self.block=nn.Sequential(nn.Conv2d(ci,co,3,padding=1,bias=False),nn.BatchNorm2d(co),nn.ReLU(inplace=True),
                                 nn.Conv2d(co,co,3,padding=1,bias=False),nn.BatchNorm2d(co),nn.ReLU(inplace=True))
    def forward(self,x):return self.block(x)

class AttentionGate(nn.Module):
    def __init__(self,skip_ch,gate_ch,mid_ch):
        super().__init__()
        self.wskip=nn.Conv2d(skip_ch,mid_ch,1,bias=False)
        self.wgate=nn.Conv2d(gate_ch,mid_ch,1,bias=False)
        self.psi=nn.Sequential(nn.ReLU(inplace=True),nn.Conv2d(mid_ch,1,1),nn.Sigmoid())
    def forward(self,skip,gate):
        g=F.interpolate(gate,size=skip.shape[-2:],mode="bilinear",align_corners=False)
        return skip*self.psi(self.wskip(skip)+self.wgate(g))

class Up(nn.Module):
    def __init__(self,cin,cskip,cout,attention=False):
        super().__init__()
        self.gate=AttentionGate(cskip,cin,max(8,cout//2)) if attention else None
        self.conv=DoubleConv(cin+cskip,cout)
    def forward(self,x,skip):
        x=F.interpolate(x,size=skip.shape[-2:],mode="bilinear",align_corners=False)
        if self.gate is not None:skip=self.gate(skip,x)
        return self.conv(torch.cat([x,skip],dim=1))

class FAKEDMultiTask(nn.Module):
    def __init__(self,backbone="resnet18",attention=False,pretrained=True):
        super().__init__()
        self.encoder,ch=make_backbone(backbone,pretrained)
        self.cls_head=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(.3),nn.Linear(ch[-1],2))
        self.up4=Up(ch[4],ch[3],128,attention)
        self.up3=Up(128,ch[2],96,attention)
        self.up2=Up(96,ch[1],64,attention)
        self.up1=Up(64,ch[0],32,attention)
        self.out=nn.Conv2d(32,1,1)
    def forward(self,x,return_seg=True):
        feats=self.encoder(x)
        a,b,c,d,e=[feats[f"s{i}"] for i in range(1,6)]
        cls=self.cls_head(e)
        if not return_seg:
            return cls,None
        # Stop noisy pseudo-mask supervision from changing classification features.
        a,b,c,d,e=[v.detach() for v in (a,b,c,d,e)]
        seg=self.up1(self.up2(self.up3(self.up4(e,d),c),b),a)
        seg=F.interpolate(self.out(seg),size=x.shape[-2:],mode="bilinear",align_corners=False)
        return cls,seg
    def param_groups(self,lr_encoder,lr_head,weight_decay):
        return [{"params":self.encoder.parameters(),"lr":lr_encoder,"weight_decay":weight_decay},
                {"params":list(self.cls_head.parameters())+list(self.up4.parameters())+
                          list(self.up3.parameters())+list(self.up2.parameters())+
                          list(self.up1.parameters())+list(self.out.parameters()),
                 "lr":lr_head,"weight_decay":weight_decay}]

# Shape-test all five architectures without fetching pretrained weights.
for name,(backbone,attention) in MODEL_SPECS.items():
    probe=FAKEDMultiTask(backbone,attention,pretrained=False).eval()
    with torch.no_grad():a,b=probe(torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE))
    assert a.shape==(1,2) and b.shape==(1,1,IMAGE_SIZE,IMAGE_SIZE),(name,a.shape,b.shape)
    print(name,"shape OK","parameters",sum(p.numel() for p in probe.parameters()))
    del probe

class ConvBlock(nn.Module):
    def __init__(self, cin, cout, negative_slope=0.01):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout),
            nn.LeakyReLU(negative_slope, inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout),
            nn.LeakyReLU(negative_slope, inplace=True))

    def forward(self, x):
        return self.net(x)


class MultiTaskNet(nn.Module):
    """Shared ResNet-18 encoder -> (a) real/fake classifier, (b) U-Net manipulation-mask decoder."""

    def __init__(self, pretrained=True):
        super().__init__()
        bb = self._backbone(pretrained)
        self.stem   = nn.Sequential(bb.conv1, bb.bn1, bb.relu)   # /2   64
        self.pool   = bb.maxpool                                  # /4
        self.layer1 = bb.layer1                                   # /4   64
        self.layer2 = bb.layer2                                   # /8  128
        self.layer3 = bb.layer3                                   # /16 256
        self.layer4 = bb.layer4                                   # /32 512

        self.cls_head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                      nn.Dropout(0.3), nn.Linear(512, 2))

        self.up4 = ConvBlock(512 + 256, 256)
        self.up3 = ConvBlock(256 + 128, 128)
        self.up2 = ConvBlock(128 + 64, 64)
        self.up1 = ConvBlock(64 + 64, 32)
        self.seg_out = nn.Conv2d(32, 1, 1)

    @staticmethod
    def _backbone(pretrained):
        w = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        return torchvision.models.resnet18(weights=w)

    @staticmethod
    def _up(x, ref):
        return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)

    def forward(self, x, return_seg=True):
        s0 = self.stem(x)
        c1 = self.layer1(self.pool(s0))
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)

        cls_logits = self.cls_head(c4)
        if not return_seg:
            return cls_logits, None
        s0,c1,c2,c3,c4=[v.detach() for v in (s0,c1,c2,c3,c4)]

        d = self.up4(torch.cat([self._up(c4, c3), c3], 1))
        d = self.up3(torch.cat([self._up(d,  c2), c2], 1))
        d = self.up2(torch.cat([self._up(d,  c1), c1], 1))
        d = self.up1(torch.cat([self._up(d,  s0), s0], 1))
        seg_logits = F.interpolate(self.seg_out(d), size=x.shape[-2:],
                                   mode="bilinear", align_corners=False)
        return cls_logits, seg_logits

    def param_groups(self, lr_encoder, lr_head, weight_decay):
        enc, head = [], []
        for name, p in self.named_parameters():
            (enc if name.split(".")[0] in
             {"stem", "layer1", "layer2", "layer3", "layer4"} else head).append(p)
        return [{"params": enc,  "lr": lr_encoder, "weight_decay": weight_decay},
                {"params": head, "lr": lr_head,    "weight_decay": weight_decay}]



# M1 faithfully uses the uploaded V2 original ResNet-18 + LeakyReLU decoder.
def create_model(name, pretrained=True):
    if name == 'M1_ResNet18_UNet':
        return MultiTaskNet(pretrained=pretrained)
    backbone, attention = MODEL_SPECS[name]
    return FAKEDMultiTask(backbone, attention, pretrained=pretrained)
assert set(MODEL_TO_RUN).issubset(MODEL_SPECS), "Unknown model name in MODEL_TO_RUN"


In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
fig,ax=plt.subplots(figsize=(15,6.3))
ax.set_xlim(0,15);ax.set_ylim(0,6.3);ax.axis('off')
def node(x,y,w,h,label,fc='#EFF5FA'):
    patch=FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.08,rounding_size=0.08',
                         linewidth=2.7,edgecolor='#172A3A',facecolor=fc)
    ax.add_patch(patch)
    ax.text(x+w/2,y+h/2,label,ha='center',va='center',fontsize=12,fontweight='bold')
def arrow(a,b):
    ax.add_patch(FancyArrowPatch(a,b,arrowstyle='-|>',mutation_scale=18,linewidth=2.9,color='#172A3A'))
node(.4,2.6,2.2,1.1,'Real / fake\nface crop')
node(3.1,2.6,2.7,1.1,'Backbone encoder\n(ResNet / EfficientNet\n/ MobileNet)')
node(6.4,4.3,2.6,1.05,'Global avg pool\n+ classification head')
node(6.4,1.0,2.6,1.4,'U-Net decoder\n(+ attention gates\nin M3 and M4)')
node(9.7,4.3,2.35,1.05,'Real / fake\nprobabilities','#EAF7ED')
node(9.7,1.0,2.35,1.4,'Pixel-wise\nprobability mask','#EAF7ED')
node(12.7,2.6,1.9,1.1,'Evaluation\n+ PNG/CSV','#FFF2D6')
arrow((2.6,3.15),(3.1,3.15));arrow((5.8,3.15),(6.25,4.75));arrow((5.8,3.15),(6.25,1.7))
arrow((9.0,4.83),(9.7,4.83));arrow((9.0,1.7),(9.7,1.7))
arrow((12.05,4.83),(13.45,3.7));arrow((12.05,1.7),(13.45,2.6))
ax.set_title('F.A.K.E.D. — Shared Multi-task Classification and Manipulation Localization',fontsize=15,fontweight='bold',pad=12)
fig.tight_layout();fig.savefig(REPORT_DIR/'FAKED_ARCHITECTURE_DIAGRAM.png',dpi=220,bbox_inches='tight')
fig.savefig(REPORT_DIR/'FAKED_ARCHITECTURE_DIAGRAM.svg',bbox_inches='tight')
plt.show();plt.close(fig)


## 8. Training losses, validation metrics and inference helpers

Loss = label-smoothed cross-entropy + weighted localization BCE and Dice. Fake crops contribute localization Dice only when their aligned pseudo-mask passes quality checks. Real crops contribute an empty-mask BCE target. We select the checkpoint by **validation AUC**, tune both decision thresholds on the validation split, and evaluate only afterward on the untouched test split.


In [ ]:
def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits.float()).flatten(1)
    t = target.float().flatten(1)
    inter = (p * t).sum(1)
    return (1 - (2 * inter + eps) / (p.sum(1) + t.sum(1) + eps)).mean()


def combined_loss(cls_logits, labels, seg_logits, masks, seg_valid, seg_weight=SEG_WEIGHT):
    ce = F.cross_entropy(cls_logits, labels, label_smoothing=LABEL_SMOOTHING)
    parts={"ce":float(ce.detach()),"bce":0.,"dice":0.}
    if seg_logits is None or seg_weight <= 0:
        return ce, parts

    has_mask=(seg_valid > .5) & (labels==1)
    is_real=(labels==0)
    seg_total=ce.new_zeros(())
    # Train on valid fake pseudo-masks. Invalid masks are excluded, never negatives.
    if has_mask.any():
        fake_bce=F.binary_cross_entropy_with_logits(seg_logits[has_mask].float(),masks[has_mask].float())
        seg_total=seg_total+fake_bce
        parts["bce"]+=float(fake_bce.detach())
        if DICE_WEIGHT>0:
            dl=dice_loss(seg_logits[has_mask].float(),masks[has_mask].float())
            seg_total=seg_total+DICE_WEIGHT*dl
            parts["dice"]=float(dl.detach())
    # Penalize positives on genuine faces explicitly, not mixed into a huge
    # pixel-averaged background batch where they may be underweighted.
    if is_real.any():
        real_bce=F.binary_cross_entropy_with_logits(seg_logits[is_real].float(),torch.zeros_like(seg_logits[is_real].float()))
        seg_total=seg_total+REAL_MASK_BCE_WEIGHT*real_bce
        parts["bce"]+=float(real_bce.detach())
    return ce+seg_weight*seg_total,parts

def iou_dice(pred, true, eps=1e-6):
    p, t = pred.astype(bool), true.astype(bool)
    inter = np.logical_and(p, t).sum()
    return ((inter + eps) / (np.logical_or(p, t).sum() + eps),
            (2 * inter + eps) / (p.sum() + t.sum() + eps))


@torch.no_grad()
def evaluate(model, loader, tta=False, mask_thr=0.5, collect_seg=False):
    """Frame-level probabilities plus segmentation IoU/Dice on masked fakes.
    `collect_seg=True` also returns raw (prob-map, gt) pairs so the mask threshold
    can be tuned without a second forward pass."""
    model.eval()
    labels_all, probs_all, ious, dices, seg_pairs = [], [], [], [], []
    real_coverage = []
    real_seg_probs=[]
    for imgs, labels, masks, seg_valid in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with amp_ctx():
            cls_logits, seg_logits = model(imgs)
            if tta:
                c2, s2 = model(torch.flip(imgs, dims=[3]))
                cls_logits = (cls_logits + c2) / 2
                seg_logits = (seg_logits + torch.flip(s2, dims=[3])) / 2

        probs_all.extend(torch.softmax(cls_logits.float(), 1)[:, 1].cpu().numpy().tolist())
        labels_all.extend(labels.numpy().tolist())

        seg_p = torch.sigmoid(seg_logits.float()).cpu().numpy()
        mk, sv, lb = masks.numpy(), seg_valid.numpy(), labels.numpy()
        for j in range(len(sv)):
            if lb[j] == 0:                        # how much mask do we wrongly paint on reals?
                real_coverage.append(float((seg_p[j, 0] > mask_thr).mean()))
                if collect_seg:real_seg_probs.append(seg_p[j,0].astype(np.float16))
                continue
            if sv[j] < 0.5:
                continue                          # fake, but no trustworthy mask -> not scored
            gt = mk[j, 0] > 0.5
            i_, d_ = iou_dice((seg_p[j, 0] > mask_thr), gt)
            ious.append(i_); dices.append(d_)
            if collect_seg:
                seg_pairs.append((seg_p[j, 0].astype(np.float16), gt))

    return {"labels": np.array(labels_all), "probs": np.array(probs_all),
            "mean_iou": float(np.mean(ious)) if ious else float("nan"),
            "mean_dice": float(np.mean(dices)) if dices else float("nan"),
            "n_seg": len(ious), "seg_pairs": seg_pairs,"real_seg_probs":real_seg_probs,
            "real_mask_coverage": float(np.mean(real_coverage)) if real_coverage else float("nan")}


def tune_mask_threshold(seg_pairs, real_seg_probs, grid=np.arange(.30,.951,.05)):
    """Validation-only mask threshold; penalize painting innocent faces.
    Uses raw fake pseudo-mask overlap minus real-face mask coverage; honest
    fake-IoU may be lower but overlays will be more clinically/visually sensible.
    """
    if not seg_pairs:return .5
    best=(float('-inf'),.5)
    for t in grid:
        fake_iou=float(np.mean([iou_dice(p.astype(np.float32)>t,g)[0] for p,g in seg_pairs]))
        real_false=float(np.mean([(p.astype(np.float32)>t).mean() for p in real_seg_probs])) if real_seg_probs else 0.
        score=fake_iou-1.2*real_false
        if score>best[0]:best=(score,float(t))
    return best[1]

def tune_threshold(labels, probs):
    """Balanced accuracy on validation only, breaking ties toward 0.5.
    Unlike using the first argmax, this avoids a spurious ~0.05 threshold when
    the model outputs uninformative scores.
    """
    grid=np.linspace(.20,.80,61)
    scores=np.array([balanced_accuracy_score(labels,(probs>=t).astype(int)) for t in grid])
    best=np.flatnonzero(np.isclose(scores,scores.max()))
    return float(grid[best[np.argmin(np.abs(grid[best]-.5))]])

def clf_metrics(labels, probs, thr=0.5):
    preds = (probs > thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = float("nan")
    return {"threshold": thr, "accuracy": accuracy_score(labels, preds),
            "balanced_acc": balanced_accuracy_score(labels, preds),
            "precision": p, "recall": r, "f1": f1, "auc": auc}


def to_video_level(df, probs):
    """Average frame probabilities within each clip (a source video's real or fake version)."""
    g = (pd.DataFrame({"clip_id": df["clip_id"].values, "label": df["label"].values, "prob": probs})
         .groupby("clip_id", as_index=False).agg(label=("label", "first"), prob=("prob", "mean")))
    return g["label"].values, g["prob"].values


print("Losses and metrics ready.")

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, precision_recall_curve

def validation_loss(model,loader, seg_weight):
    model.eval();total=0.;count=0
    with torch.no_grad():
        for x,y,m,valid in loader:
            x=x.to(DEVICE);y=y.to(DEVICE);m=m.to(DEVICE);valid=valid.to(DEVICE)
            with amp_ctx(): cl,seg=model(x)
            loss,_=combined_loss(cl.float(),y,seg.float(),m,valid,seg_weight=seg_weight)
            total+=float(loss)*len(x);count+=len(x)
    return total/max(1,count)

def train_model(model,name,outdir):
    optimizer=torch.optim.AdamW(model.param_groups(LR_ENCODER,LR_HEAD,WEIGHT_DECAY))
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
    steps=max(1,len(train_dl));total_steps=max(1,EPOCHS*steps);warm_steps=max(1,WARMUP_EPOCHS*steps)
    def scale(step):
        if step<warm_steps: return (step+1)/warm_steps
        t=(step-warm_steps)/max(1,total_steps-warm_steps)
        return 0.5*(1+math.cos(math.pi*min(1.0,t)))
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,scale)
    history=[];best_auc=-float('inf');best_epoch=0;start=time.time();ckpt=outdir/'best_model.pt'
    best_secondary=-float('inf')
    for epoch in range(1,EPOCHS+1):
        if time.time()-start>TRAIN_TIME_BUDGET_SEC and epoch>1:
            print(name,': per-model training time cap reached');break
        model.train();epoch_loss=0.;n_train=0;correct=0
        # Stage 1: pure deepfake classification. Stage 2: gradually train
        # segmentation decoder against (noisy) pseudo-mask targets.
        seg_weight=(0.0 if epoch<=CLS_ONLY_EPOCHS else SEG_WEIGHT*min(1.0,(epoch-CLS_ONLY_EPOCHS)/3.0))
        for x,y,m,valid in tqdm(train_dl,desc=f'{name} epoch {epoch}',leave=False):
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            m=m.to(DEVICE,non_blocking=True);valid=valid.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with amp_ctx():cl,seg=model(x,return_seg=(seg_weight>0))
            loss,_=combined_loss(cl.float(),y,seg.float() if seg is not None else None,m,valid,seg_weight=seg_weight)
            if not torch.isfinite(loss):raise RuntimeError(f'Nonfinite training loss in {name}')
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            scaler.step(optimizer);scaler.update();scheduler.step()
            epoch_loss+=float(loss.item())*len(y);n_train+=len(y)
            correct+=int((cl.detach().argmax(1)==y).sum().item())
        val=evaluate(model,val_dl,tta=False)
        vm=clf_metrics(val['labels'],val['probs'],0.5)
        auc=float(vm['auc']);vloss=validation_loss(model,val_dl,seg_weight)
        va_vid_y,va_vid_p=to_video_level(val_df,val['probs'])
        va_vid_bal=balanced_accuracy_score(va_vid_y,(va_vid_p>=.5).astype(int))
        history.append({'epoch':epoch,'train_loss':epoch_loss/max(1,n_train),
                        'val_loss':vloss,'train_accuracy':correct/max(1,n_train),
                        'val_accuracy':float(vm['accuracy']),'val_auc':auc,'val_video_balanced_accuracy':float(va_vid_bal),'seg_loss_weight':seg_weight,
                        'val_pseudo_mask_iou':float(val['mean_iou'])})
        pd.DataFrame(history).to_csv(outdir/'epoch_history.csv',index=False)
        improved=(np.isfinite(auc) and (auc>best_auc+1e-4 or
                  (abs(auc-best_auc)<=1e-4 and va_vid_bal>best_secondary+1e-4)))
        if improved:
            best_auc=auc;best_secondary=float(va_vid_bal);best_epoch=epoch
            torch.save({'state_dict':model.state_dict(),'name':name,'epoch':epoch,'run_id':RUN_ID},ckpt)
        print(f'{name} | epoch={epoch} train_loss={history[-1]["train_loss"]:.4f} '
              f'val_loss={vloss:.4f} val_acc={vm["accuracy"]:.3f} val_auc={auc:.3f}')
        if epoch>=MIN_TRAIN_EPOCHS and epoch-best_epoch>=EARLY_STOP_PATIENCE:
            print('Early stopping:',name);break
    if not ckpt.exists():raise RuntimeError(f'No checkpoint for {name}; inspect split and training.')
    model.load_state_dict(torch.load(ckpt,map_location=DEVICE,weights_only=True)['state_dict'])
    return pd.DataFrame(history),float(time.time()-start)

# Fixed, held-out test IDs MUST be the same for all architecture figures.
fakes=test_df[(test_df.label==1)&(test_df.seg_valid==1)].sample(
    n=min(4,int(((test_df.label==1)&(test_df.seg_valid==1)).sum())),random_state=SEED)
reals=test_df[test_df.label==0].sample(n=min(2,int((test_df.label==0).sum())),random_state=SEED)
fixed_examples=pd.concat([fakes,reals]).reset_index(drop=True)
assert len(fakes)>0 and len(reals)>0, 'Need real and masked fake test images; increase sample count.'
fixed_examples.to_csv(REPORT_DIR/'SAME_HELD_OUT_TEST_IMAGES.csv',index=False)

@torch.no_grad()
def save_prediction_examples(model,outdir,mask_thr,cls_thr):
    model.eval();probabilities=[];mask_probs=[]
    for _,r in fixed_examples.iterrows():
        im=cv2.imread(str(r.image_path))
        if im is None:raise FileNotFoundError(str(r.image_path))
        rgb=cv2.cvtColor(im,cv2.COLOR_BGR2RGB).astype(np.float32)/255.
        im_norm=(rgb-IMAGENET_MEAN)/IMAGENET_STD
        x=torch.from_numpy(im_norm.transpose(2,0,1).copy()).unsqueeze(0).to(DEVICE)
        with amp_ctx():cl,seg=model(x)
        if ENABLE_TTA:
            with amp_ctx():cl2,seg2=model(torch.flip(x,dims=[3]))
            cl=(cl+cl2)/2;seg=(seg+torch.flip(seg2,dims=[3]))/2
        probabilities.append(float(torch.softmax(cl.float(),dim=1)[0,1]))
        mask_probs.append(torch.sigmoid(seg.float())[0,0].cpu().numpy())
    np.savez_compressed(outdir/'same_input_predictions.npz',prob_fake=np.asarray(probabilities),
                        mask_prob=np.stack(mask_probs),mask_threshold=float(mask_thr),cls_threshold=float(cls_thr))

@torch.no_grad()
def benchmark_inference(model,n=15):
    model.eval();x=torch.randn(1,3,IMAGE_SIZE,IMAGE_SIZE,device=DEVICE)
    for _ in range(3):model(x)
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    t=time.perf_counter()
    for _ in range(n):model(x)
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    return 1000.*(time.perf_counter()-t)/n



def refine_segmentation_decoder(model, name, outdir):
    if SEG_REFINE_EPOCHS <= 0:
        return 0.
    t0=time.time()
    for p in model.parameters():p.requires_grad_(False)
    seg_names=['up1','up2','up3','up4','seg_out' if name=='M1_ResNet18_UNet' else 'out']
    decoder=[getattr(model,k) for k in seg_names]
    decoder_params=[]
    for layer in decoder:
        for p in layer.parameters():p.requires_grad_(True);decoder_params.append(p)
    opt=torch.optim.AdamW(decoder_params,lr=SEG_REFINE_LR,weight_decay=WEIGHT_DECAY)
    seg_rows=[];best_score=-float('inf');best_path=outdir/'best_seg_refined.pt'
    for ep in range(1,SEG_REFINE_EPOCHS+1):
        model.eval()  # freeze feature encoder BN statistics
        for layer in decoder:layer.train()
        n=0;sum_loss=0.
        for x,y,m,valid in train_dl:
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            m=m.to(DEVICE,non_blocking=True);valid=valid.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp_ctx():cls,seg=model(x)
            has_fake=((y==1) & (valid>.5));has_real=(y==0)
            segloss=seg.float().sum()*0.0
            if has_fake.any():
                segloss=segloss+F.binary_cross_entropy_with_logits(seg[has_fake].float(),m[has_fake].float())
                segloss=segloss+DICE_WEIGHT*dice_loss(seg[has_fake],m[has_fake])
            if has_real.any():
                segloss=segloss+REAL_MASK_BCE_WEIGHT*F.binary_cross_entropy_with_logits(
                    seg[has_real].float(),torch.zeros_like(seg[has_real].float()))
            if not (has_fake.any() or has_real.any()):continue
            segloss.backward()
            nn.utils.clip_grad_norm_(decoder_params,GRAD_CLIP)
            opt.step()
            sum_loss+=float(segloss.detach())*len(x);n+=len(x)
        val=evaluate(model,val_dl,tta=False,collect_seg=True)
        mask_thr=tune_mask_threshold(val['seg_pairs'],val['real_seg_probs'])
        fake_iou=np.mean([iou_dice(p.astype(np.float32)>mask_thr,g)[0] for p,g in val['seg_pairs']]) if val['seg_pairs'] else 0.
        real_fraction=np.mean([(p.astype(np.float32)>mask_thr).mean() for p in val['real_seg_probs']]) if val['real_seg_probs'] else 0.
        score=float(fake_iou-1.2*real_fraction)
        seg_rows.append({'seg_epoch':ep,'train_seg_loss':sum_loss/max(1,n),
                         'validation_pseudo_iou':float(fake_iou),
                         'validation_real_false_mask_area':float(real_fraction),
                         'validation_mask_threshold':float(mask_thr),'validation_seg_score':score})
        print(f'{name} | seg refine {ep}/{SEG_REFINE_EPOCHS}: '
              f'val pseudo IoU {fake_iou:.3f}, real false coverage {real_fraction:.3f}')
        if score>best_score:
            best_score=score
            torch.save(model.state_dict(),best_path)
    if best_path.exists():
        model.load_state_dict(torch.load(best_path,map_location=DEVICE,weights_only=True))
        # This checkpoint preserves the chosen classification weights and the
        # validation-selected refined decoder, independent of test data.
        torch.save({'state_dict':model.state_dict(),'name':name,'run_id':RUN_ID,
                    'classification_preserved':True,'segmentation_refined':True},outdir/'best_model.pt')
    pd.DataFrame(seg_rows).to_csv(outdir/'SEGMENTATION_REFINEMENT_CURVES.csv',index=False)
    fig,ax=plt.subplots(figsize=(6,4))
    ax.plot([r['seg_epoch'] for r in seg_rows],[r['validation_pseudo_iou'] for r in seg_rows],label='Validation fake pseudo-IoU')
    ax.plot([r['seg_epoch'] for r in seg_rows],[r['validation_real_false_mask_area'] for r in seg_rows],label='Validation real false-mask coverage')
    ax.set(xlabel='Segmentation refinement epoch',ylabel='Fraction / IoU',title=name+' decoder refinement')
    ax.legend();fig.tight_layout();fig.savefig(outdir/'SEGMENTATION_REFINEMENT.png',dpi=180);plt.close(fig)
    for p in model.parameters():p.requires_grad_(True)
    model.eval()
    return float(time.time()-t0)

results=[]
for name in MODEL_TO_RUN:
    backbone,attention=MODEL_SPECS[name]
    outdir=REPORT_DIR/name;outdir.mkdir(exist_ok=True)
    metrics_path=outdir/'metrics.json'
    if metrics_path.exists() and (outdir/'same_input_predictions.npz').exists() and (outdir/'best_model.pt').exists():
        old=json.loads(metrics_path.read_text())
        if old.get('run_id')==RUN_ID:
            print('RESUME — already complete:',name);results.append(old);continue
    print('\n'+'='*65+'\nTRAINING',name,'\n'+'='*65)
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    if DEVICE.type=='cuda':torch.cuda.manual_seed_all(SEED)
    model=create_model(name,pretrained=USE_PRETRAINED).to(DEVICE)
    hist,duration=train_model(model,name,outdir)
    seg_refine_seconds=refine_segmentation_decoder(model,name,outdir)
    val=evaluate(model,val_dl,tta=ENABLE_TTA,collect_seg=True)
    cls_thr=tune_threshold(val['labels'],val['probs'])
    mask_thr=tune_mask_threshold(val['seg_pairs'],val['real_seg_probs'])
    test=evaluate(model,test_dl,tta=ENABLE_TTA,mask_thr=mask_thr)
    fmet=clf_metrics(test['labels'],test['probs'],cls_thr)
    video_y,video_p=to_video_level(test_df,test['probs'])
    val_video_y,val_video_p=to_video_level(val_df,val['probs'])
    video_thr=tune_threshold(val_video_y,val_video_p)
    vmet=clf_metrics(video_y,video_p,video_thr)
    yp=(test['probs']>=cls_thr).astype(int)
    cm=confusion_matrix(test['labels'],yp,labels=[0,1]);np.savetxt(outdir/'confusion_matrix_counts.csv',cm,fmt='%d',delimiter=',')
    fig,ax=plt.subplots(figsize=(5,4));ax.imshow(cm,cmap='Blues')
    for (i,j),n in np.ndenumerate(cm):ax.text(j,i,str(n),ha='center',va='center',fontweight='bold')
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Real','Fake'],yticklabels=['Real','Fake'],
           xlabel='Predicted class',ylabel='Actual class',title=name+'\nTest confusion matrix')
    fig.tight_layout();fig.savefig(outdir/'confusion_matrix.png',dpi=200,bbox_inches='tight');plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    axes[0].plot(hist.epoch,hist.train_loss,'o-',label='Train loss')
    axes[0].plot(hist.epoch,hist.val_loss,'o-',label='Validation loss')
    axes[0].legend();axes[0].set(title='Train/validation loss',xlabel='Epoch',ylabel='Loss')
    axes[1].plot(hist.epoch,hist.train_accuracy,'o-',label='Train accuracy')
    axes[1].plot(hist.epoch,hist.val_accuracy,'o-',label='Validation accuracy')
    axes[1].plot(hist.epoch,hist.val_auc,'--',label='Validation ROC-AUC')
    axes[1].set(title='Train/validation accuracy and AUC',xlabel='Epoch',ylabel='Score',ylim=(0,1.05))
    axes[1].legend(fontsize=9);fig.suptitle(name);fig.tight_layout()
    fig.savefig(outdir/'ACCURACY_AND_LOSS_CURVES.png',dpi=200,bbox_inches='tight');plt.close(fig)
    fig,ax=plt.subplots(figsize=(5,4))
    if len(np.unique(test['labels']))==2:
        fpr,tpr,_=roc_curve(test['labels'],test['probs']);ax.plot(fpr,tpr,label=f'AUC={fmet["auc"]:.3f}')
    ax.plot([0,1],[0,1],'--',alpha=.55);ax.legend();ax.set(xlabel='FPR',ylabel='TPR',title=name+' test ROC')
    fig.tight_layout();fig.savefig(outdir/'ROC_CURVE.png',dpi=180);plt.close(fig)
    fig,ax=plt.subplots(figsize=(5.7,4))
    ax.hist(test['probs'][test['labels']==0],bins=20,alpha=.6,label='Real (test)')
    ax.hist(test['probs'][test['labels']==1],bins=20,alpha=.6,label='Fake (test)')
    ax.axvline(cls_thr,color='black',linestyle='--',label='Threshold (validation)')
    ax.set(xlabel='P(fake)',ylabel='Number of frames',title=name+' score distribution')
    ax.legend();fig.tight_layout()
    fig.savefig(outdir/'PREDICTION_PROBABILITY_DISTRIBUTION.png',dpi=180);plt.close(fig)
    save_prediction_examples(model,outdir,mask_thr,cls_thr)
    pd.DataFrame({'sample_id':test_df.sample_id,'clip_id':test_df.clip_id,'label':test_df.label,
                  'method':test_df.manipulation_method,'prob_fake':test['probs'],'predicted_label':yp,
                  'image_path':test_df.image_path}).to_csv(outdir/'TEST_PREDICTIONS.csv',index=False)
    method_metrics=[]
    for method,group in test_df[test_df.label==1].groupby('manipulation_method'):
        indices=group.index.values
        method_metrics.append({'manipulation_method':method,'n_fake_test_frames':len(group),
                               'fake_recall':float(np.mean(yp[indices]==1))})
    pd.DataFrame(method_metrics).to_csv(outdir/'BY_MANIPULATION_METHOD.csv',index=False)
    try:latency=benchmark_inference(model)
    except Exception as err:print('Inference benchmark unavailable:',err);latency=float('nan')
    metrics={'run_id':RUN_ID,'model':name,'backbone':backbone,'attention_decoder':bool(attention),
             'test_frame_accuracy':float(fmet['accuracy']),'test_frame_balanced_accuracy':float(fmet['balanced_acc']),
             'test_frame_precision':float(fmet['precision']),'test_frame_recall':float(fmet['recall']),
             'test_frame_f1':float(fmet['f1']),'test_frame_roc_auc':float(fmet['auc']),
             'test_video_accuracy':float(vmet['accuracy']),'test_video_roc_auc':float(vmet['auc']),
             'test_pseudo_mask_dice':float(test['mean_dice']),'test_pseudo_mask_iou':float(test['mean_iou']),
             'test_real_false_mask_area':float(test['real_mask_coverage']),
             'n_test_frames':len(test_df),'n_test_clips':len(video_y),
             'n_scored_fake_pseudo_masks':int(test['n_seg']),
             'training_protocol':'classification first, detached encoder for segmentation, mild augmentation',
             'classification_threshold_tuned_on_validation':cls_thr,
             'video_threshold_tuned_on_validation':video_thr,
             'mask_threshold_tuned_on_validation':mask_thr,
             'parameters':sum(p.numel() for p in model.parameters()),'inference_ms_per_frame':latency,
             'train_seconds':duration,'seg_refine_seconds':seg_refine_seconds,'epochs_completed':len(hist)}
    metrics_path.write_text(json.dumps(metrics,indent=2))
    results.append(metrics)
    pd.DataFrame(results).to_csv(REPORT_DIR/'ALL_FIVE_MODELS_COMPARISON.csv',index=False)
    print(pd.Series(metrics).to_string())
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()

comparison=pd.DataFrame(results)
display(comparison[['model','test_frame_accuracy','test_video_accuracy',
                    'test_frame_f1','test_pseudo_mask_dice','test_pseudo_mask_iou']].round(4))


In [ ]:
model_names=list(MODEL_TO_RUN)
predictions={name:np.load(REPORT_DIR/name/'same_input_predictions.npz') for name in model_names}
assert all(len(predictions[n]['mask_prob'])==len(fixed_examples) for n in model_names)
for plot_type in ['heatmap','overlay']:
    fig,axes=plt.subplots(len(fixed_examples),len(model_names)+2,
                          figsize=(3.0*(len(model_names)+2),3.05*len(fixed_examples)),squeeze=False)
    for i,(_,row) in enumerate(fixed_examples.iterrows()):
        raw=cv2.cvtColor(cv2.imread(str(row.image_path)),cv2.COLOR_BGR2RGB)
        axes[i,0].imshow(raw)
        axes[i,0].set_title(f'INPUT: {"FAKE" if int(row.label) else "REAL"}',fontsize=11,fontweight='bold')
        if bool(row.seg_valid) and isinstance(row.mask_path,str) and row.mask_path:
            gt=cv2.imread(str(row.mask_path),cv2.IMREAD_GRAYSCALE)
        else:gt=np.zeros(raw.shape[:2],dtype=np.uint8)
        axes[i,1].imshow(gt,cmap='gray',vmin=0,vmax=255)
        axes[i,1].set_title('Pseudo-reference mask',fontsize=10,fontweight='bold')
        for j,name in enumerate(model_names,2):
            pred=predictions[name]['mask_prob'][i]
            thr=float(predictions[name]['mask_threshold'])
            if plot_type=='heatmap':
                axes[i,j].imshow(pred,cmap='inferno',vmin=0,vmax=1)
            else:
                overlay=raw.copy()
                region=pred>thr
                overlay[region]=(0.55*overlay[region]+0.45*np.array([255,0,0])).astype(np.uint8)
                axes[i,j].imshow(overlay)
            p=float(predictions[name]['prob_fake'][i])
            axes[i,j].set_title(name.replace('_',' ')+'\nP(fake)='+f'{p:.2f}',fontsize=8,fontweight='bold')
        for a in axes[i]:a.axis('off')
    fig.suptitle('F.A.K.E.D. — Same held-out inputs vs five '+plot_type+' predictions (overlays class-gated)' ,fontsize=14,fontweight='bold',y=0.995)
    fig.tight_layout(rect=[0,0,1,0.975])
    out=REPORT_DIR/('SAME_INPUT_FIVE_MODEL_HEATMAPS.png' if plot_type=='heatmap'
                    else 'SAME_INPUT_FIVE_MODEL_SEGMENTATION_OVERLAYS.png')
    fig.savefig(out,dpi=180,bbox_inches='tight')
    plt.show();plt.close(fig)
# Extra individual model segmentation panels for readable report closeups.
for name in model_names:
    arr=predictions[name]
    fig,axes=plt.subplots(len(fixed_examples),3,figsize=(9,2.8*len(fixed_examples)),squeeze=False)
    for i,(_,row) in enumerate(fixed_examples.iterrows()):
        raw=cv2.cvtColor(cv2.imread(str(row.image_path)),cv2.COLOR_BGR2RGB)
        heat=arr['mask_prob'][i];region=(heat>float(arr['mask_threshold'])) & (float(arr['prob_fake'][i])>=float(arr['cls_threshold']))
        overlay=raw.copy();overlay[region]=(0.55*overlay[region]+0.45*np.array([255,0,0])).astype(np.uint8)
        for j,(im,title,cm) in enumerate([(raw,'Input face',None),(heat,'Predicted mask heatmap','inferno'),(overlay,'Segmented overlay',None)]):
            axes[i,j].imshow(im,cmap=cm,vmin=0 if j==1 else None,vmax=1 if j==1 else None)
            axes[i,j].set_title(title,fontsize=11,fontweight='bold');axes[i,j].axis('off')
    fig.suptitle(name,fontsize=14,fontweight='bold');fig.tight_layout()
    fig.savefig(REPORT_DIR/name/'INDIVIDUAL_FACE_OUTPUTS.png',dpi=170,bbox_inches='tight');plt.close(fig)


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,4.8))
for ax,key,title in [(axes[0],'test_frame_accuracy','Frame classification accuracy'),
                     (axes[1],'test_frame_f1','Frame fake-class F1'),
                     (axes[2],'test_pseudo_mask_iou','Pseudo-mask IoU')]:
    ax.bar(np.arange(len(comparison)),comparison[key])
    ax.set(xticks=np.arange(len(comparison)),xticklabels=[str(x).split('_')[0] for x in comparison.model],
           ylim=(0,1.05),ylabel='Held-out test metric',title=title)
    for i,v in enumerate(comparison[key]):
        ax.text(i,float(v)+0.02,f'{float(v):.3f}',ha='center',fontsize=9)
fig.suptitle('F.A.K.E.D. — Five-architecture comparison',fontsize=14,fontweight='bold')
fig.tight_layout();fig.savefig(REPORT_DIR/'FIVE_MODEL_RESULTS_COMPARISON.png',dpi=200,bbox_inches='tight');plt.show();plt.close(fig)

experiment_manifest={'run_id':RUN_ID,'source_dataset':'FaceForensics++ C23, attached Kaggle mirror',
    'indexed_dataset_mp4_count':len(video_manifest),'indexed_original_mp4_count':len(real_manifest),
    'indexed_manipulated_mp4_count':len(fake_manifest),
    'requested_source_pairs':int(N_SOURCE_VIDEOS),'chosen_source_pairs_before_preprocessing':len(pairs),
    'actually_processed_source_pairs':int(frame_manifest.video_id.nunique()),
    'requested_frames_per_pair':int(FRAMES_PER_VIDEO),'actual_crop_count':len(frame_manifest),
    'actual_real_crop_count':int((frame_manifest.label==0).sum()),
    'actual_fake_crop_count':int((frame_manifest.label==1).sum()),
    'actual_valid_fake_pseudo_masks':int(frame_manifest.seg_valid.sum()),
    'split_table':split_counts.to_dict('records'),'models_run':model_names,
    'img_size':int(IMAGE_SIZE),'augmentation':'training only: random horizontal flip and mild brightness/contrast (no blur/JPEG/zoom)',
    'mask_source':'aligned frame differences, morphology and mask area rejection — pseudo-reference masks',
    'training_protocol':'classification-only warmup; decoder supervision from pseudo-masks with detached encoder; threshold penalizes real false-mask area',
    'pretrained':USE_PRETRAINED,'smoke_test':SMOKE_TEST,
    'reporting_note':'Results were obtained on the stated experimental sample, NOT the full dataset.'}
(REPORT_DIR/'EXPERIMENT_AND_DATASET_FACTS.json').write_text(json.dumps(experiment_manifest,indent=2))
# One-page summary suitable for writing §5 and §6, not a fabricated conclusion.
winner=sorted(results,key=lambda r:(-r['test_frame_f1'],-r['test_pseudo_mask_iou']))[0]
lines=['F.A.K.E.D. EXPERIMENT — ACTUAL MEASUREMENTS','',
       f'Experiment type: {"SMOKE TEST — not reportable" if SMOKE_TEST else "Computationally bounded source-disjoint experiment"}',
       f'Indexed original videos: {len(real_manifest)}; indexed manipulated videos: {len(fake_manifest)}',
       f'Actual processed source pairs: {frame_manifest.video_id.nunique()} / requested {N_SOURCE_VIDEOS}',
       f'Actual real/fake crops: {(frame_manifest.label==0).sum()} / {(frame_manifest.label==1).sum()}',
       f'Valid fake pseudo-masks: {int(frame_manifest.seg_valid.sum())}',
       '', 'Measured architecture results:']
for r in results:
    lines.append(f'{r["model"]}: frame acc {r["test_frame_accuracy"]:.4f}; F1 {r["test_frame_f1"]:.4f}; '
                 f'video acc {r["test_video_accuracy"]:.4f}; pseudo-IoU {r["test_pseudo_mask_iou"]:.4f}')
lines+=['',f'Highest test frame F1: {winner["model"]} ({winner["test_frame_f1"]:.4f})',
        'Warning: method selection on held-out test set is descriptive, not evidence of generalization.',
        'Localization masks are difference-derived pseudo-labels, not manual annotations.',
        'The notebook outputs do not verify published journal literature; references must be checked independently.',
        'There is no guaranteed 85% result; numerical results are computed from actual predictions.']
(REPORT_DIR/'REPORT_RESULTS_README.txt').write_text('\n'.join(lines))
print('\n'.join(lines))
import zipfile
archive=WORK_ROOT/'FAKED_REPORT_RESULTS.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for f in REPORT_DIR.rglob('*'):
        if f.is_file() and f.suffix!='.pt':z.write(f,f.relative_to(REPORT_DIR))
    # representative real/fake crops and their pseudo-reference masks
    for sid in sample_ids:
        rows=frame_manifest[frame_manifest.sample_id==sid]
        for _,r in rows.iterrows():
            img=Path(r.image_path)
            if img.exists():z.write(img,arcname=f'example_face_crops/{"real" if r.label==0 else "fake"}/{img.name}')
            if isinstance(r.mask_path,str) and r.mask_path and Path(r.mask_path).exists():
                mp=Path(r.mask_path);z.write(mp,arcname=f'example_face_crops/pseudo_masks/{mp.name}')
print('\nALL DONE. Download from Kaggle output:')
print(archive)
print('Model .pt checkpoints:',REPORT_DIR)


In [ ]:
# Machine-readable summary for Section 5 / Section 6 of the report.
cols=['model','test_frame_accuracy','test_video_accuracy','test_frame_f1',
      'test_frame_roc_auc','test_pseudo_mask_dice','test_pseudo_mask_iou',
      'test_real_false_mask_area','epochs_completed','train_seconds']
ranked=comparison[cols].sort_values(['test_video_accuracy','test_frame_accuracy'],ascending=False)
ranked.to_csv(REPORT_DIR/'REPORT_TABLE_READY.csv',index=False)
print('FINAL MODEL RANKING (test set; descriptive comparison only):')
display(ranked.round(4))
print('\nACTUAL EXPERIMENT SUBSET:', int(frame_manifest.video_id.nunique()), 'paired source videos; ',len(frame_manifest),'face crops')
print('Above 85% frame accuracy?:', bool((comparison.test_frame_accuracy>=.85).any()))
print('Above 85% video accuracy?:', bool((comparison.test_video_accuracy>=.85).any()))
print('Download ZIP:',archive)
print('Check every epoch_history.csv for train vs validation curves before drawing conclusions.')
import zipfile
with zipfile.ZipFile(archive,'a',zipfile.ZIP_DEFLATED) as z:
    z.write(REPORT_DIR/'REPORT_TABLE_READY.csv',arcname='REPORT_TABLE_READY.csv')
